# 7. Validation Review Policy (Colab)

This notebook applies the reviewed keep/flag/exclude policy to a completed TCN run.

It should be run **after** `6_TCN_Training_Colab.ipynb`, once a `predictions.csv` artifact exists for the selected run.

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

## Configure Run

Set `RUN_NAME` to the TCN run you want to review.

In [ ]:
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection')
RUN_NAME = 'squat_tcn_l1_channels96'

PREDICTIONS_CSV = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / RUN_NAME / 'predictions.csv'
REVIEW_CSV = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/validation_failure_review.csv'
POLICY_SCRIPT = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/apply_validation_review_policy.py'

print('RUN_NAME =', RUN_NAME)
print('PREDICTIONS_CSV =', PREDICTIONS_CSV)
print('REVIEW_CSV =', REVIEW_CSV)
print('POLICY_SCRIPT =', POLICY_SCRIPT)
print('predictions exists =', PREDICTIONS_CSV.exists())
print('review exists =', REVIEW_CSV.exists())
print('script exists =', POLICY_SCRIPT.exists())

## Apply Policy

This exports:

- `policy_filtered_metrics_summary.json`
- `policy_filtered_valid_predictions.csv`

next to the selected `predictions.csv`.

In [ ]:
!python {POLICY_SCRIPT} \
  --predictions-csv {PREDICTIONS_CSV} \
  --review-csv {REVIEW_CSV}

## Load Summary

In [ ]:
import json

SUMMARY_JSON = PREDICTIONS_CSV.with_name('policy_filtered_metrics_summary.json')
print('SUMMARY_JSON =', SUMMARY_JSON)

summary = json.loads(SUMMARY_JSON.read_text())
summary

## Inspect Filtered Predictions

In [ ]:
import pandas as pd

FILTERED_CSV = PREDICTIONS_CSV.with_name('policy_filtered_valid_predictions.csv')
filtered_df = pd.read_csv(FILTERED_CSV)
filtered_df.sort_values('abs_error', ascending=False)

## Optional Quick Comparison

In [ ]:
before = summary['valid_metrics_before_policy']
after = summary['valid_metrics_after_policy']

comparison_df = pd.DataFrame([
    {'view': 'before_policy', **before},
    {'view': 'after_policy', **after},
])
comparison_df